In [0]:
from pyspark.sql.functions import *

In [0]:
bronze02_df = spark.read.table(
    "grocery_delivery_project.bronze.bronze_df02"
)

In [0]:
silver02_df=(
    bronze02_df

    .withColumn(
        "customer_name",
        upper(trim(col("customer_name")))
    )
     .withColumn(
        "city",
        upper(trim(col("city")))
    )
      .withColumn(
        "product_name",
        upper(trim(col("product_name")))
    )
      
       .withColumn(
        "net_sales",
        round(
            (col("unit_price") * col("quantity")) - col("discount"), 2)
        )
)

In [0]:
display(silver02_df)

In [0]:
from pyspark.sql.functions import *

# Read Batch 2 again from Bronze
bronze02_df = spark.read.table(
    "grocery_delivery_project.bronze.bronze_df02"
)

# Rebuild Silver Batch 2 from scratch
silver02_df = (
    bronze02_df
    .withColumn(
        "order_date",
        coalesce(
            expr("try_to_date(cast(order_date as string), 'yyyy-MM-dd')"),
            expr("try_to_date(cast(order_date as string), 'M/d/yyyy')")
        )
    )
    .withColumn("customer_name", upper(trim(col("customer_name"))))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("product_name", upper(trim(col("product_name"))))
    .withColumn(
        "net_sales",
        round(
            (col("unit_price") * col("quantity")) - col("discount"),
            2
        )
    )
)

display(silver02_df)
silver02_df.printSchema()

In [0]:
initial_silver_df = spark.read.table(
    "grocery_delivery_project.silver.silver_table"
)

In [0]:
new_records_df = silver02_df.join(
    initial_silver_df,
    on="order_id",
    how="left_anti"
)

display(new_records_df)

In [0]:
print("New records count:", new_records_df.count())

In [0]:
silver02_df.write\
    .mode("overwrite")\
        .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("grocery_delivery_project.silver.cleaned_incremental")

In [0]:
existing_records_df = silver02_df.join(
    initial_silver_df,
    on="order_id",
    how="inner"
)

display(existing_records_df)

In [0]:
print("Existing records count:", existing_records_df.count())

In [0]:
city_changes_df = (
    silver02_df.alias("new")
    .join(
        initial_silver_df.alias("old"),
        on="order_id",
        how="inner"
    )
    .filter(
        col("new.city") != col("old.city")
    )
    .select(
        col("order_id"),
        col("old.city").alias("old_city"),
        col("new.city").alias("new_city")
    )
)

display(city_changes_df)

In [0]:
def compatibility_check(target_df,source_df):
    target_schema={
        field.name:field.dataType.simpleString()
        for field in target_df.schema.fields
    }
    source_schema={
        field.name:field.dataType.simpleString()
        for field in source_df.schema.fields
    }

    if target_schema==source_schema:
        return True
    else:
        print("false")
        print(target_schema)
        print(source_schema)
        return False
       



In [0]:
compatibility_check(
    initial_silver_df, silver02_df
)

In [0]:
from pyspark.sql.functions import *

# Read Batch 2 again from Bronze
bronze02_df = spark.read.table(
    "grocery_delivery_project.bronze.bronze_df02"
)

# Rebuild Silver Batch 2 from scratch
silver02_df = (
    bronze02_df
    .withColumn(
        "order_date",
        coalesce(
            expr("try_to_date(cast(order_date as string), 'yyyy-MM-dd')"),
            expr("try_to_date(cast(order_date as string), 'M/d/yyyy')")
        )
    )
    .withColumn("customer_name", upper(trim(col("customer_name"))))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("product_name", upper(trim(col("product_name"))))
    .withColumn(
        "net_sales",
        round(
            (col("unit_price") * col("quantity")) - col("discount"),
            2
        )
    )
)

display(silver02_df)
silver02_df.printSchema()

In [0]:
silver02_df.select(
    "order_id",
    "order_date"
).display()

In [0]:
compatibility_check(
    initial_silver_df,
    silver02_df
)

In [0]:
from pyspark.sql.functions import *

invalid_records_df = silver02_df.filter(
    col("order_id").isNull() |
    col("order_date").isNull() |
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("city").isNull() |
    col("product_name").isNull() |
    col("category").isNull() |
    col("quantity").isNull() |
    col("unit_price").isNull() |
    col("discount").isNull() |
    col("delivery_status").isNull()
)

display(invalid_records_df)
print("Invalid records:", invalid_records_df.count())

In [0]:
from pyspark.sql.functions import *

blank_records_df = silver02_df.filter(
    (trim(col("order_id")) == "") |
    (trim(col("customer_id")) == "") |
    (trim(col("customer_name")) == "") |
    (trim(col("city")) == "") |
    (trim(col("product_name")) == "") |
    (trim(col("category")) == "") |
    (trim(col("delivery_status")) == "")
)

display(blank_records_df)
print("Blank records:", blank_records_df.count())

In [0]:
duplicate_records_df = (
    silver02_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_records_df)
print("Duplicate order IDs:", duplicate_records_df.count())

In [0]:
valid_source_df = silver02_df

initial_silver_df = spark.read.table(
    "grocery_delivery_project.silver.silver_table"
)

# New records
new_records_df = valid_source_df.join(
    initial_silver_df,
    on="order_id",
    how="left_anti"
)

# Existing records
existing_records_df = valid_source_df.join(
    initial_silver_df,
    on="order_id",
    how="inner"
)

# City changes
city_changes_df = (
    valid_source_df.alias("new")
    .join(initial_silver_df.alias("old"), on="order_id", how="inner")
    .filter(col("new.city") != col("old.city"))
    .select(
        col("order_id"),
        col("old.city").alias("old_city"),
        col("new.city").alias("new_city")
    )
)

print("New records:", new_records_df.count())
print("Existing records:", existing_records_df.count())
print("City changes:", city_changes_df.count())

display(new_records_df)
display(city_changes_df)